In [ ]:
import cadquery as cq
from jupyter_cadquery import *
from jupyter_cadquery.replay import replay, enable_replay

enable_replay(show_bbox=True, warning=False)
show_object = replay

# UNITS ARE MM 

# initial cylinder parameters
R = 50
H = 500
fillet_r = 30

# initial cylinder 
cyl = cq.Workplane("front").circle(R).extrude(H)
cyl_fillet = cyl.edges("front").fillet(fillet_r).edges("back").fillet(fillet_r)

# add inlet and outlet for inner fluid
inner_fluid_r = 30
inner_fluid_depth = 50

inlet_outlet = cyl_fillet.edges("back").circle(inner_fluid_r).extrude(-inner_fluid_depth).edges("front").circle(inner_fluid_r).extrude(inner_fluid_depth)

# add inlet and outlet for outer fluid
outer_fluid_r = 15
outer_fluid_depth = 100
outer_fluid_inlet_z = H*0.8 - H/2 # 80 percent of the cylinders length
outer_fluid_outlet_z = H*0.2- H/2 # 20 percent of the cylinders length

of_inlet_wp = (
    inlet_outlet
    .workplane(offset=outer_fluid_inlet_z)             # move to middle of cylinder length
    .transformed(rotate=(90, 0, 0))    # rotate plane to face the cylinder wall
    .center(0, 0)                      # move outward to the shell surface
)

of_outlet_wp = (
    inlet_outlet
    .workplane(offset=outer_fluid_outlet_z)             # move to middle of cylinder length
    .transformed(rotate=(270, 0, 0))    # rotate plane to face the cylinder wall
    .center(0, 0)                      # move outward to the shell surface
)

of_inlet = of_inlet_wp.circle(outer_fluid_r).extrude(outer_fluid_depth)
of_outlet = of_outlet_wp.circle(outer_fluid_r).extrude(outer_fluid_depth)

inner_outer_cyl = inlet_outlet+ of_inlet + of_outlet

# tag surfaces
inner_outer_cyl.faces("+Z").tag("inner_outlet")
inner_outer_cyl.faces("-Z").tag("inner_inlet")
inner_outer_cyl.faces("+Y").tag("outer_inlet")
inner_outer_cyl.faces("-Y").tag("outer_outlet")

def generate_hole_positions(holes_per_row, col_spacing, row_spacing):
    """
    holes_per_row: list of ints, e.g. [3, 4, 3]
    col_spacing: spacing between columns
    row_spacing: spacing between rows
    """
    holes = []
    n_rows = len(holes_per_row)

    # center rows vertically around y=0
    row_offsets = [
        (i - (n_rows - 1) / 2) * row_spacing
        for i in range(n_rows)
    ]

    for row_idx, n_holes in enumerate(holes_per_row):
        y = row_offsets[row_idx]

        # center columns horizontally
        col_offsets = [
            (i - (n_holes - 1) / 2) * col_spacing
            for i in range(n_holes)
        ]

        for x in col_offsets:
            holes.append((x, y))

    return holes

# baffle parameters
baffle_radius = R
baffle_thickness = 10
baffle_start = fillet_r + H/20
offset = 0 #3*R
baffle_end = H - H/20 - fillet_r

# hole parameters
hole_diameter = 15
col_spacing = baffle_radius / 2     # auto-tied to baffle size
row_spacing = baffle_radius / 2
holes_per_row = [3, 4, 3]           # change to anything you want

# generate hole positions
holes = generate_hole_positions(
    holes_per_row,
    col_spacing=col_spacing,
    row_spacing=row_spacing
)

baffle = (
    cq.Workplane("front")
    .circle(baffle_radius)
    .extrude(baffle_thickness)
)

baffle1 = baffle.translate((offset, 0, baffle_start))
baffle2 = baffle.translate((offset, 0, baffle_end))

pipes = []

dz = baffle_end - baffle_start + baffle_thickness

# make a pipe at each hole position
for (x, y) in holes:
    pipe = (
        cq.Workplane("front")
        .workplane(offset=baffle_start)
        .center(x, y)
        .circle(hole_diameter/2)     # same size as the holes or slightly smaller
        .extrude(dz)                 # goes to the 2nd baffle
    )
    pipes.append(pipe)

# combine unshelled pipes into one object for cutting baffles
pipes_for_cut = pipes[0]  
for p in pipes[1:]:
    pipes_for_cut = pipes_for_cut.union(p)

pipes_for_cut = pipes_for_cut.translate((offset, 0, 0))
num_semi_baffles = 4   
pipe_radius = hole_diameter / 2

dz = baffle_end - baffle_start
semi_spacing = dz / (num_semi_baffles + 1)
num_semi_baffles = 4  
pipe_radius = hole_diameter / 2

dz = baffle_end - baffle_start
semi_spacing = dz / (num_semi_baffles + 1)

def make_semi_baffle(index, z_pos):
    """
    function to make semi-baffles (non-circular) for body of HX
    index = 0,1,2,... (for alternating pattern)
    z_pos = absolute z location
    """

    # start with circular baffle
    b = (
        cq.Workplane("front")
        .workplane(offset=z_pos)
        .circle(baffle_radius)
        .extrude(baffle_thickness)
    )

    # MAKE CUT SHAPE
    # annular region between inner pipe radius and outer baffle radius
    annulus = (
        cq.Workplane("front")
        .workplane(offset=z_pos)
        .circle(baffle_radius)      # outer
        # .circle(pipe_radius)        # inner
        .extrude(baffle_thickness)
    )

    # define half-plane cut direction
    cut_selector = cq.selectors.BoxSelector

    if index % 2 == 0:
        # EVEN index → remove UPPER section
        cut_box = (
            cq.Workplane("front")
            .workplane(offset=z_pos)
            .rect(2*baffle_radius, 2*baffle_radius)
            .extrude(baffle_thickness)
            .translate((0, baffle_radius/2, 0))  # upper half
        )
    else:
        # ODD index → remove LOWER section
        cut_box = (
            cq.Workplane("front")
            .workplane(offset=z_pos)
            .rect(2*baffle_radius, 2*baffle_radius)
            .extrude(baffle_thickness)
            .translate((0, -baffle_radius/2, 0))  # lower half
        )

    # intersect annulus with half-plane box to get the semi-shape
    cut_shape = annulus.intersect(cut_box)

    # subtract it from the full baffle
    semi_baffle = cut_shape   # keep only the half-annulus


    return semi_baffle
    
semi_baffles = []

for i in range(num_semi_baffles):
    z = baffle_start + (i + 1) * semi_spacing
    sb = make_semi_baffle(i, z)
    cut_sb = sb.cut(pipes_for_cut)
    semi_baffles.append(cut_sb)

semi_group = cq.Compound.makeCompound([sb.val() for sb in semi_baffles])
semi_group = semi_group.translate((offset, 0, 0))

# CUT PIPES FROM BAFFLES
baffle1 = baffle1.cut(pipes_for_cut)
baffle2 = baffle2.cut(pipes_for_cut)    

# now shell pipes 
pipes_all = pipes[0].faces(">Z or <Z").shell(-1)  
for p in pipes[1:]:
    shelled = p.faces(">Z or <Z").shell(-1)  # make them hollow
    pipes_all = pipes_all.union(shelled)

pipes_all = pipes_all.translate((offset, 0, 0))

# combined inner wall structure
wall_structure = baffle1 + baffle2 + pipes_all + semi_group

# now need to cut wall structure from inner_outer_cyl 
fluid_volumes = inner_outer_cyl.cut(wall_structure)
# TODO: figure out how to tag two volumes and export as .stl

hx = cq.Assembly() #  WHATEVER IS ADDED TO ASSEMBLY IS VISUALIZED BELOW

hx.add(wall_structure, name="wall_structure", color=cq.Color("gray"))
hx.add(fluid_volumes, name="fluid_volumes", color=cq.Color("green", alpha=0.3))
hx.save("hx.step")




Enabling jupyter_cadquery replay
c


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

ConnectionError: HTTPConnectionPool(host='localhost', port=8888): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8888): Failed to establish a new connection: [Errno 61] Connection refused"))

In [ ]:
import cadquery as cq
from jupyter_cadquery import show, set_defaults
from jupyter_cadquery.replay import replay, enable_replay

enable_replay()
show_object = replay

# ---------------------------
# PARAMETERS
# ---------------------------
# Cylinder
R = 50                   # cylinder radius
H = 500                  # cylinder height
fillet_r = 30            # end fillet radius

if fillet_r >= R:
    print("Fillet radius cannot be bigger than cylinder radius")
    
# Inner fluid
inner_fluid_r = 30       # diameter of inner pipe holes
inner_fluid_depth = 50   # extrusion depth for inner fluid

# Outer fluid
outer_fluid_r = 15
outer_fluid_depth = 100
outer_fluid_inlet_frac = 0.8  # fraction along cylinder length
outer_fluid_outlet_frac = 0.2

# Baffles
baffle_thickness = 10
baffle_start_offset = fillet_r + H/20
baffle_end_offset = H - H/20 - fillet_r
offset = 3 * R           # translation along X-axis

# Baffle hole pattern
holes_per_row = [3, 4, 3]
hole_diameter = 15
col_spacing = R / 2
row_spacing = R / 2

# Semi-baffles
num_semi_baffles = 6

# ---------------------------
# CYLINDER
# ---------------------------
cyl = cq.Workplane("front").circle(R).extrude(H)
cyl_fillet = cyl.edges("front").fillet(fillet_r).edges("back").fillet(fillet_r)

# Inner fluid inlet/outlet
inlet_outlet = (
    cyl_fillet.edges("back")
    .circle(inner_fluid_r).extrude(-inner_fluid_depth)
    .edges("front").circle(inner_fluid_r).extrude(inner_fluid_depth)
)

# Outer fluid inlet/outlet
outer_inlet_z = H * outer_fluid_inlet_frac - H / 2
outer_outlet_z = H * outer_fluid_outlet_frac - H / 2

of_inlet = (
    inlet_outlet
    .workplane(offset=outer_inlet_z)
    .transformed(rotate=(90, 0, 0))
    .center(0, 0)
    .circle(outer_fluid_r).extrude(outer_fluid_depth)
)

of_outlet = (
    inlet_outlet
    .workplane(offset=outer_outlet_z)
    .transformed(rotate=(270, 0, 0))
    .center(0, 0)
    .circle(outer_fluid_r).extrude(outer_fluid_depth)
)

inner_outer_cyl = inlet_outlet.union(of_inlet).union(of_outlet)

# ---------------------------
# FUNCTION: GENERATE HOLE POSITIONS
# ---------------------------
def generate_hole_positions(holes_per_row, col_spacing, row_spacing):
    holes = []
    n_rows = len(holes_per_row)
    row_offsets = [(i - (n_rows - 1)/2) * row_spacing for i in range(n_rows)]
    for row_idx, n_holes in enumerate(holes_per_row):
        y = row_offsets[row_idx]
        col_offsets = [(i - (n_holes - 1)/2) * col_spacing for i in range(n_holes)]
        for x in col_offsets:
            holes.append((x, y))
    return holes

holes = generate_hole_positions(holes_per_row, col_spacing, row_spacing)

# ---------------------------
# BAFFLES
# ---------------------------
baffle_base = (
    cq.Workplane("front")
    .circle(R)
    .extrude(baffle_thickness)
    .faces(">Z")
    .workplane()
    .pushPoints(holes)
    .hole(hole_diameter)
)

baffle1 = baffle_base.translate((offset, 0, baffle_start_offset))
baffle2 = baffle_base.translate((offset, 0, baffle_end_offset))

# ---------------------------
# PIPES (INNER FLUID)
# ---------------------------
dz = baffle_end_offset - baffle_start_offset + baffle_thickness
pipes = []
for (x, y) in holes:
    pipe = (
        cq.Workplane("front")
        .workplane(offset=baffle_start_offset)
        .center(x, y)
        .circle(hole_diameter/2)
        .extrude(dz)
    )
    pipes.append(pipe)

pipes_all = pipes[0]
for p in pipes[1:]:
    pipes_all = pipes_all.union(p)

pipes_all = pipes_all.translate((offset, 0, 0))

# ---------------------------
# SEMI-BAFFLES
# ---------------------------
pipe_radius = hole_diameter / 2
semi_spacing = (baffle_end_offset - baffle_start_offset) / (num_semi_baffles + 1)

def make_semi_baffle(index, z_pos):
    # Full circular baffle
    b = cq.Workplane("front").workplane(offset=z_pos).circle(R).extrude(baffle_thickness)
    # Annular cut
    annulus = cq.Workplane("front").workplane(offset=z_pos).circle(R).circle(pipe_radius).extrude(baffle_thickness)
    if index % 2 == 0:
        cut_box = cq.Workplane("front").workplane(offset=z_pos).rect(2*R, 2*R).extrude(baffle_thickness).translate((0, R/2, 0))
    else:
        cut_box = cq.Workplane("front").workplane(offset=z_pos).rect(2*R, 2*R).extrude(baffle_thickness).translate((0, -R/2, 0))
    semi_baffle = annulus.intersect(cut_box)  # keep only half-annulus
    return semi_baffle

semi_baffles = [make_semi_baffle(i, baffle_start_offset + (i+1)*semi_spacing) for i in range(num_semi_baffles)]
semi_group = cq.Compound.makeCompound([sb.val() for sb in semi_baffles])
semi_group = semi_group.translate((offset, 0, 0))

# ---------------------------
# SHOW FINAL ASSEMBLY
# ---------------------------
show(
    inner_outer_cyl,
    baffle1,
    baffle2,
    pipes_all,
    semi_group
)
